# 01 · LeNet on MNIST —— CNN 家族起点：给网络装上"眼睛"

**家族位置**：`02_CNN_Family` 第 1 个项目（后续：`02_AlexNet_VGG_FashionMNIST` → `03_ResNet_CIFAR10` → ...）

**与 01 家族的衔接**：MLP 把 28×28 展平成 784 维向量，"看到"的只是像素统计——它没有局部感受野，也没有平移不变性。LeNet（LeCun, 1998）是第一个"看图"的网络：直接在二维图上滑动卷积核。本项目的任务很直接：**用 MLP 1/4 的参数量，超过它 98.14% 的成绩**。

**学习目标**
1. 理解卷积/池化带来的三大归纳偏置：局部感受野、权值共享、平移等变
2. 逐层推演 LeNet-5 的尺寸与参数量（并在代码里逐层打印验证）
3. 可视化卷积特征图——亲眼看到 CNN 把一张图"拆"成了什么
4. 与 01 家族 MLP 正面对比：参数量 / 精度 / 错误数

## 1. 原理：CNN 的三大归纳偏置

### 为什么 MLP 不适合图像

展平 28×28 变成 784 维向量：图像里的"相邻像素构成笔画"这个结构**被抹掉了**；而且输入稍微大一点（如 224×224），全连接层的参数量就爆炸到不可训练。

### ① 局部感受野（Local Receptive Field）

每个神经元只连接输入的一个小邻域（如 5×5）。图像的局部相关性很强——一个像素与它 5 个像素外的点几乎无关，卷积核天然符合这个先验。

### ② 权值共享（Weight Sharing）

同一个卷积核**扫过整张图**（等价于对全图做模板匹配）。好处：参数量从"输入×输出"降到"核大小"；同时获得**平移等变**——数字"7"出现在左上角还是右下角，识别方式一样。

### ③ 池化降采样（Pooling）

对 2×2 邻域取平均/最大值，分辨率减半：计算量下降、感受野逐层变大（浅层看笔画 → 深层看结构）、对小位移更鲁棒。

### LeNet-5 结构（现代复现版）

```
输入 28×28×1
 → Conv 5×5×6 (pad=2) → ReLU → AvgPool 2×2     6×14×14
 → Conv 5×5×16       → ReLU → AvgPool 2×2     16×5×5
 → Flatten (400) → FC 120 → FC 84 → FC 10
```

原版（1998）用 tanh + RBF 输出层，且输入是 32×32；现代复现改成 ReLU + Linear + CrossEntropy，在 MNIST 上效果更好。

### 逐层参数量演算（合计 61,706）

| 层 | 参数量 | 推导 |
|---|---|---|
| conv1 6@5×5 | 156 | (5×5×1 + 1) × 6 |
| conv2 16@5×5 | 2,416 | (5×5×6 + 1) × 16 |
| fc1 400→120 | 48,120 | 400×120 + 120 |
| fc2 120→84 | 10,164 | 120×84 + 84 |
| fc3 84→10 | 850 | 84×10 + 10 |
| **合计** | **61,706** | 对照：MLP 是 235,146（3.8 倍） |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_mnist_torch
from common.engine import fit
from common.models import LeNet
from common.utils import count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：MNIST（通道优先，不展平）

与 01 家族同一份数据、同一套标准化（0.1307/0.3081），唯一区别：保留 (N, 1, 28, 28) 的四维结构——这是 CNN 的输入格式。

In [ ]:
DATA_ROOT = ROOT / "data"
Xtr, ytr, Xte, yte = load_mnist_torch(str(DATA_ROOT))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)
print("类别分布(训练集):", np.bincount(ytr.numpy()).tolist())

fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i in range(10):
    idx = int(np.where(ytr.numpy() == i)[0][0])
    axes[i].imshow(Xtr[idx, 0], cmap="gray")
    axes[i].axis("off")
plt.suptitle("每个类别各取一个样本（CNN 输入格式：通道优先）", fontsize=11)
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型：LeNet-5 + 逐层尺寸/参数验证

定义在 `common/models.py`。下面让一张图走一遍完整前向，逐层打印输出尺寸——和原理节的手算表格对答案；参数总量用 `count_params` 复核对 61,706。

In [ ]:
model = LeNet()
print(model)
print(f"可训练参数量: {count_params(model):,}（MLP 是 235,146，本模型只有其 {count_params(model) / 235146:.1%}）")

x = Xte[:1]
print("\n逐层尺寸演算（输入", tuple(x.shape), "）:")
for i, m in enumerate(model.net):
    x = m(x)
    print(f"  [{i:02d}] {m.__class__.__name__:12s} → {tuple(x.shape)}")

batch_size = 128
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=512)

## 4. 训练

与 01 家族 MLP 完全同规格：`Adam(lr=1e-3)`、batch 128、10 epochs、seed 0（结果可复现）。CPU 上约 1 分钟。

In [ ]:
hist = fit(model, train_loader, test_loader, epochs=10, lr=1e-3, device=DEVICE)
print(f"\n最终测试准确率: {hist['val_acc'][-1]:.2%}")

## 5. 结果解读：曲线与混淆矩阵

套路与 01 家族 02 项目一致：先看 val 曲线是否健康，再看错误集中在哪里。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(hist["train_loss"], label="train", marker="o", ms=3)
axes[0].plot(hist["val_loss"], label="val", marker="o", ms=3)
axes[0].set_title("Loss 曲线"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist["train_acc"], label="train", marker="o", ms=3)
axes[1].plot(hist["val_acc"], label="val", marker="o", ms=3)
axes[1].set_title("Accuracy 曲线"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xte)
    pred = logits.argmax(1)

acc = (pred == yte).float().mean().item()
print(f"测试集准确率: {acc:.2%}")
print(f"答错 {int((1 - acc) * len(yte))} / {len(yte)}（MLP 答错 186）")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title("混淆矩阵")
plt.colorbar(im)
plt.savefig(FIGS / "fig2_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
pairs = [(int(r), int(c), int(cm_off[r, c])) for idx in flat for r, c in [np.unravel_index(idx, cm.shape)]]
print("最常被看错的 5 对（真实→预测）:", pairs)

### 错误样本

同样挑 10 个答错的，对比 01 家族 02 项目那批——CNN 的"盲区"是否更接近人眼？

In [ ]:
wrong = torch.where(pred != yte)[0]
print(f"测试集共答错 {len(wrong)} 个（错误率 {(1 - acc):.2%}）")

fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for ax_, idx in zip(axes.ravel(), wrong[:10]):
    ax_.imshow(Xte[idx, 0], cmap="gray")
    ax_.set_title(f"真:{yte[idx].item()} 预:{pred[idx].item()}", fontsize=10)
    ax_.axis("off")
plt.suptitle("前 10 个错误样本", fontsize=12)
plt.savefig(FIGS / "fig3_errors.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 特征图可视化：CNN 看到了什么

把一张"7"送进网络，取出 conv1 的 6 张特征图和 conv2 的 16 张特征图。浅层特征图保留原始形状（笔画、拐角），深层在更小的分辨率上组合出"结构响应"——这就是"浅层学笔画、深层学结构"的直观证据。

In [ ]:
model.eval()
idx = int(np.where(yte == 7)[0][0])
img = Xte[idx]
with torch.no_grad():
    conv1 = model.net[:2](img.unsqueeze(0))   # Conv+ReLU → (1, 6, 28, 28)
    conv2 = model.net[:5](img.unsqueeze(0))   # +Pool+Conv+ReLU → (1, 16, 10, 10)

fig, axes = plt.subplots(2, 16, figsize=(13, 3.6))
axes[0, 0].imshow(img[0], cmap="gray"); axes[0, 0].set_title("输入", fontsize=8); axes[0, 0].axis("off")
for j in range(6):
    axes[0, 1 + j].imshow(conv1[0, j], cmap="viridis"); axes[0, 1 + j].axis("off")
for j in range(6, 15):
    axes[0, 1 + j].axis("off")
for j in range(16):
    axes[1, j].imshow(conv2[0, j], cmap="viridis"); axes[1, j].axis("off")
axes[0, 0].set_ylabel("conv1\n6×28×28", fontsize=8)
axes[1, 0].set_ylabel("conv2\n16×10×10", fontsize=8)
plt.suptitle("同一张 7 的卷积特征图（viridis 配色）", fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / "fig4_featuremaps.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 与 01 家族 MLP 的正面对比

同数据、同超参、同 10 epochs，唯一变量是架构。若 LeNet 用更少参数拿到更高精度，就证明**归纳偏置在替我们省参数**。

In [ ]:
lenet_params = count_params(model)
lenet_errors = int((1 - acc) * len(yte))
mlp = {"params": 235146, "acc": 0.9814, "errors": 186}  # 01 家族 02 项目记录值

print(f"{'架构':10s} {'参数量':>9s} {'测试准确率':>10s} {'答错':>6s}")
print(f"{'MLP':10s} {mlp['params']:>9,} {mlp['acc']:>10.2%} {mlp['errors']:>6}")
print(f"{'LeNet-5':10s} {lenet_params:>9,} {acc:>10.2%} {lenet_errors:>6}")
print(f"\n参数量 LeNet/MLP = {lenet_params / mlp['params']:.1%}，错误数 LeNet/MLP = {lenet_errors / mlp['errors']:.1%}")
assert acc > mlp["acc"], "LeNet 应超过 MLP"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
names = ["MLP", "LeNet-5"]
axes[0].bar(names, [mlp["params"], lenet_params], color=["#4C72B0", "#DD8452"])
axes[0].set_yscale("log")
axes[0].set_title("可训练参数量（log 轴）"); axes[0].set_ylabel("参数个数")
for i, v in enumerate([mlp["params"], lenet_params]):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
axes[1].bar(names, [mlp["acc"], acc], color=["#4C72B0", "#DD8452"])
axes[1].set_ylim(0.97, 0.995)
axes[1].set_title("MNIST 测试准确率"); axes[1].set_ylabel("accuracy")
for i, v in enumerate([mlp["acc"], acc]):
    axes[1].text(i, v, f"{v:.2%}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig5_mlp_vs_lenet.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. 总结与下一步

**本项目收获**
1. CNN 三件套（局部感受野 / 权值共享 / 池化）把"像素统计"升级成"结构感知"——参数量降为 MLP 的 26%，精度反超
2. 逐层尺寸/参数演算 + 代码打印双向验证：手算和框架输出对上了
3. 特征图可视化给了"浅层笔画 → 深层结构"一个直观证据
4. 错误分析套路（曲线/混淆矩阵/错误样本）从 01 家族原样复用——这套评估框架跨架构通用

**下一步**：`02_AlexNet_VGG_FashionMNIST`——把网络"加深加宽"（ReLU + Dropout + 更大卷积核），体验深度带来的训练难点（梯度、过拟合、收敛速度），为 03 的 ResNet 残差思想埋伏笔。